In [3]:
print('init')

init


# Phase 1 - Data Cleaning and Station Mapping (Train 18238)

This section builds a clean, chronologically ordered dataset for train 18238 and assigns a stable halt-only station index using the static route JSON.

In [4]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

TRAIN_NO = 18238
DATA_PATH = Path("..\\dataset\\supabase\\rail_data_rows.csv")
STATIC_ROUTE_PATH = Path("../docs/statics/18238_static.json")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find dataset at: {DATA_PATH.resolve()}")
if not STATIC_ROUTE_PATH.exists():
    raise FileNotFoundError(f"Could not find static route at: {STATIC_ROUTE_PATH.resolve()}")

raw_df = pd.read_csv(DATA_PATH)

# Drop columns not needed for modeling.
cols_to_drop = ["created_at", "updated_at"]
raw_df = raw_df.drop(columns=cols_to_drop, errors="ignore")

# Filter to train 18238 and keep key columns.
train_df = raw_df[raw_df["train_no"] == TRAIN_NO].copy()
if train_df.empty:
    raise ValueError(f"No rows found for train_no={TRAIN_NO}")

# Parse journey_date as date for consistent grouping.
train_df["journey_date"] = pd.to_datetime(train_df["journey_date"], errors="coerce").dt.date
train_df = train_df.dropna(subset=["journey_date", "station_code"])

# Keep latest record per journey and station (polling can create duplicates).
if "id" in train_df.columns:
    train_df = train_df.sort_values(["journey_date", "station_code", "id"])
    train_df = train_df.drop_duplicates(subset=["journey_date", "station_code"], keep="last")

# Ensure delay_departure_minutes is numeric for modeling.
train_df["delay_departure_minutes"] = pd.to_numeric(
    train_df["delay_departure_minutes"], errors="coerce"
 )

print(f"Train rows after cleaning: {len(train_df):,}")
print(f"Journeys available: {train_df['journey_date'].nunique()}")
print(f"Stations observed: {train_df['station_code'].nunique()}")

display(train_df.head(10))

Train rows after cleaning: 9,095
Journeys available: 107
Stations observed: 85


,id,train_no,journey_date,station_code,scheduled_arrival,scheduled_departure,actual_arrival,actual_departure,delay_arrival_minutes,delay_departure_minutes,distance_from_origin_km
394,395,18238,2026-01-17,AGC,1768700700,1.768701e+09,1.768709e+09,1.768710e+09,137.0,148.0,NaN
429,430,18238,2026-01-17,AGN,1768778760,1.768779e+09,1.768781e+09,1.768781e+09,39.0,41.0,NaN
418,419,18238,2026-01-17,AMLA,1768755300,1.768755e+09,1.768757e+09,1.768757e+09,20.0,24.0,NaN
356,357,18238,2026-01-17,ASR,1768645500,1.768646e+09,NaN,1.768646e+09,NaN,16.0,NaN
402,403,18238,2026-01-17,BAB,1768722360,1.768722e+09,1.768725e+09,1.768726e+09,49.0,53.0,NaN
409,410,18238,2026-01-17,BAQ,1768734420,1.768735e+09,1.768738e+09,1.768738e+09,52.0,59.0,NaN
358,359,18238,2026-01-17,BEAS,1768647480,1.768648e+09,1.768649e+09,1.768649e+09,21.0,26.0,NaN
410,411,18238,2026-01-17,BHS,1768735980,1.768736e+09,1.768740e+09,1.768740e+09,63.0,69.0,NaN
407,408,18238,2026-01-17,BINA,1768731900,1.768732e+09,1.768735e+09,1.768735e+09,44.0,52.0,NaN
434,435,18238,2026-01-17,BPHB,1768789260,1.768789e+09,1.768790e+09,1.768790e+09,8.0,11.0,NaN


In [5]:
with open(STATIC_ROUTE_PATH, "r", encoding="utf-8") as f:
    static_json = json.load(f)

route_rows = static_json.get("data", {}).get("route", [])
halt_route = [r for r in route_rows if int(r.get("isHalt", 0)) == 1]
halt_station_order = [
    r.get("stationCode")
    for r in sorted(halt_route, key=lambda x: x.get("sequence", 10**9))
    if r.get("stationCode")
]

if len(halt_station_order) < 2:
    raise ValueError("Static route does not contain enough halt stations.")

station_index_map = {code: idx for idx, code in enumerate(halt_station_order)}

# Keep only halt stations from the static route and assign station_index.
train_df = train_df[train_df["station_code"].isin(station_index_map)].copy()
train_df["station_index"] = train_df["station_code"].map(station_index_map)

train_df = train_df.sort_values(["journey_date", "station_index"]).reset_index(drop=True)

missing_halts = [s for s in halt_station_order if s not in set(train_df["station_code"]) ]

print(f"Halt stations in static route: {len(halt_station_order)}")
print(f"Halt stations present in data: {train_df['station_code'].nunique()}")
print(f"Missing halt stations in data: {len(missing_halts)}")
if missing_halts:
    print(missing_halts)

print("\nSample ordered rows:")
display(train_df.head(12))

Halt stations in static route: 85
Halt stations present in data: 85
Missing halt stations in data: 0

Sample ordered rows:


,id,train_no,journey_date,station_code,scheduled_arrival,scheduled_departure,actual_arrival,actual_departure,delay_arrival_minutes,delay_departure_minutes,distance_from_origin_km,station_index
0,357,18238,2026-01-17,ASR,1768645500,1.768646e+09,NaN,1.768646e+09,NaN,16.0,NaN,0
1,358,18238,2026-01-17,JNL,1768646340,1.768646e+09,1.768647e+09,1.768648e+09,15.0,19.0,NaN,1
2,359,18238,2026-01-17,BEAS,1768647480,1.768648e+09,1.768649e+09,1.768649e+09,21.0,26.0,NaN,2
3,360,18238,2026-01-17,JUC,1768649880,1.768650e+09,1.768651e+09,1.768652e+09,22.0,26.0,NaN,3
4,361,18238,2026-01-17,JRC,1768651740,1.768652e+09,1.768652e+09,1.768653e+09,11.0,14.0,NaN,4
5,362,18238,2026-01-17,PGW,1768652580,1.768653e+09,1.768653e+09,1.768654e+09,13.0,17.0,NaN,5
6,363,18238,2026-01-17,PHR,1768653780,1.768654e+09,1.768655e+09,1.768655e+09,14.0,17.0,NaN,6
7,364,18238,2026-01-17,LDH,1768655100,1.768656e+09,1.768656e+09,1.768657e+09,12.0,22.0,NaN,7
8,365,18238,2026-01-17,SNL,1768658280,1.768658e+09,1.768658e+09,1.768659e+09,-1.0,3.0,NaN,8
9,366,18238,2026-01-17,DOA,1768658760,1.768659e+09,1.768659e+09,1.768659e+09,4.0,6.0,NaN,9


# Phase 1.4 - Feature Engineering (Train 18238)

This section creates ML-ready features for next-station delay prediction. Features are computed using past-only information within each journey.

In [6]:
# Build base features and target for next-station delay prediction
# Target: delay_departure_minutes at station i+1

feature_df = train_df.copy()

# Ensure chronological ordering within each journey
feature_df = feature_df.sort_values(["journey_date", "station_index"]).reset_index(drop=True)

# Core lag features within each journey
feature_df["current_delay"] = feature_df["delay_departure_minutes"]
feature_df["previous_delay_1"] = feature_df.groupby("journey_date")["delay_departure_minutes"].shift(1)
feature_df["previous_delay_2"] = feature_df.groupby("journey_date")["delay_departure_minutes"].shift(2)

# Rolling features (past-only)
feature_df["rolling_avg_last_3_delays"] = (
    feature_df.groupby("journey_date")["delay_departure_minutes"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)
feature_df["rolling_std_last_3_delays"] = (
    feature_df.groupby("journey_date")["delay_departure_minutes"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=2).std())
)

# Trend slope over last 3 delays (simple linear fit)
def _trend_slope(values: pd.Series) -> float:
    if values.isna().all():
        return np.nan
    y = values.values.astype(float)
    x = np.arange(len(y), dtype=float)
    if len(y) < 2:
        return np.nan
    # simple slope of least squares line
    x_mean = x.mean()
    y_mean = np.nanmean(y)
    denom = ((x - x_mean) ** 2).sum()
    if denom == 0:
        return np.nan
    return float(np.nansum((x - x_mean) * (y - y_mean)) / denom)

feature_df["delay_trend_slope"] = (
    feature_df.groupby("journey_date")["delay_departure_minutes"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=2).apply(_trend_slope, raw=False))
)

# Station progression features
max_station_index = feature_df["station_index"].max()
feature_df["journey_progress_pct"] = feature_df["station_index"] / max_station_index

# Temporal features from journey_date (proxy; for richer time features use actual timestamps)
feature_df["journey_date_dt"] = pd.to_datetime(feature_df["journey_date"], errors="coerce")
feature_df["day_of_week"] = feature_df["journey_date_dt"].dt.dayofweek
feature_df["month"] = feature_df["journey_date_dt"].dt.month
feature_df["is_weekend"] = feature_df["day_of_week"].isin([5, 6]).astype(int)

# Target: next station delay
feature_df["target_next_delay"] = feature_df.groupby("journey_date")["delay_departure_minutes"].shift(-1)
feature_df["next_station_index"] = feature_df.groupby("journey_date")["station_index"].shift(-1)

# Remove rows where we cannot form a next-station target
feature_df = feature_df.dropna(subset=["target_next_delay", "next_station_index"]).copy()
feature_df["next_station_index"] = feature_df["next_station_index"].astype(int)

# Basic sanity checks
print(f"Feature rows: {len(feature_df):,}")
print(f"Journeys covered: {feature_df['journey_date'].nunique()}")
print("Feature null rates (top 10):")
null_rates = feature_df.isna().mean().sort_values(ascending=False)
print(null_rates.head(10))

# Preview selected features
preview_cols = [
    "journey_date",
    "station_code",
    "station_index",
    "current_delay",
    "previous_delay_1",
    "previous_delay_2",
    "rolling_avg_last_3_delays",
    "rolling_std_last_3_delays",
    "delay_trend_slope",
    "journey_progress_pct",
    "day_of_week",
    "month",
    "is_weekend",
    "target_next_delay",
]

display(feature_df[preview_cols].head(12))

Feature rows: 8,316
Journeys covered: 107
Feature null rates (top 10):
distance_from_origin_km      1.000000
previous_delay_1             0.075517
previous_delay_2             0.075517
delay_departure_minutes      0.062771
current_delay                0.062771
actual_departure             0.062771
rolling_std_last_3_delays    0.026215
delay_trend_slope            0.026215
delay_arrival_minutes        0.019721
actual_arrival               0.019721
dtype: float64


,journey_date,station_code,station_index,current_delay,previous_delay_1,previous_delay_2,rolling_avg_last_3_delays,rolling_std_last_3_delays,delay_trend_slope,journey_progress_pct,day_of_week,month,is_weekend,target_next_delay
0,2026-01-17,ASR,0,16.0,NaN,NaN,NaN,NaN,NaN,0.000000,5,1,1,19.0
1,2026-01-17,JNL,1,19.0,16.0,NaN,16.000000,NaN,NaN,0.011905,5,1,1,26.0
2,2026-01-17,BEAS,2,26.0,19.0,16.0,17.500000,2.121320,0.75,0.023810,5,1,1,26.0
3,2026-01-17,JUC,3,26.0,26.0,19.0,20.333333,5.131601,5.00,0.035714,5,1,1,14.0
4,2026-01-17,JRC,4,14.0,26.0,26.0,23.666667,4.041452,3.50,0.047619,5,1,1,17.0
5,2026-01-17,PGW,5,17.0,14.0,26.0,22.000000,6.928203,-6.00,0.059524,5,1,1,17.0
6,2026-01-17,PHR,6,17.0,17.0,14.0,19.000000,6.244998,-4.50,0.071429,5,1,1,22.0
7,2026-01-17,LDH,7,22.0,17.0,17.0,16.000000,1.732051,1.50,0.083333,5,1,1,3.0
8,2026-01-17,SNL,8,3.0,22.0,17.0,18.666667,2.886751,2.50,0.095238,5,1,1,6.0
9,2026-01-17,DOA,9,6.0,3.0,22.0,14.000000,9.848858,-7.00,0.107143,5,1,1,9.0


In [7]:
# Additional required features (distance, temporal, segment, historical stats)
feature_df = feature_df.sort_values(["journey_date", "station_index"]).reset_index(drop=True)

# Static route metadata (halt-only) for distance/schedule features
halt_meta = {r.get("stationCode"): r for r in halt_route if r.get("stationCode")}

def _abs_minutes(meta: dict, key: str) -> float:
    if not meta:
        return np.nan
    val = meta.get(key)
    if val is None:
        return np.nan
    day = meta.get("day", 1)
    return (day - 1) * 1440 + float(val)

# Distance features
if "distance_from_origin_km" in feature_df.columns:
    feature_df["distance_from_origin"] = pd.to_numeric(
        feature_df["distance_from_origin_km"], errors="coerce"
    )
else:
    feature_df["distance_from_origin"] = np.nan

# If dataset distance is missing, use static route distance from source
if feature_df["distance_from_origin"].isna().all():
    feature_df["distance_from_origin"] = feature_df["station_code"].map(
        lambda c: halt_meta.get(c, {}).get("distanceFromSourceKm")
    )

if feature_df["distance_from_origin"].notna().any():
    max_dist = feature_df["distance_from_origin"].max()
    if pd.notna(max_dist) and max_dist != 0:
        feature_df["journey_progress_pct"] = feature_df["distance_from_origin"] / max_dist

# Static schedule features (minutes from journey start)
feature_df["scheduled_arrival_min"] = feature_df["station_code"].map(
    lambda c: _abs_minutes(halt_meta.get(c), "scheduledArrival")
 )
feature_df["scheduled_departure_min"] = feature_df["station_code"].map(
    lambda c: _abs_minutes(halt_meta.get(c), "scheduledDeparture")
 )
feature_df["halt_duration_min"] = feature_df["station_code"].map(
    lambda c: halt_meta.get(c, {}).get("haltDurationMinutes")
 )
feature_df["speed_on_section_kmph"] = feature_df["station_code"].map(
    lambda c: halt_meta.get(c, {}).get("speedOnSectionKmph")
 )

# Temporal features (hour_of_day, overnight_segment_flag)
time_candidates = [
    "actual_departure",
    "actual_arrival",
    "scheduled_departure",
    "scheduled_arrival",
    "expected_departure",
    "expected_arrival",
    "departure_time",
    "arrival_time",
    "departure",
    "arrival",
]
time_col = next((c for c in time_candidates if c in feature_df.columns), None)
if time_col:
    time_parsed = pd.to_datetime(feature_df[time_col], errors="coerce")
    if time_parsed.isna().all():
        time_parsed = pd.to_datetime(
            feature_df["journey_date"].astype(str)
            + " "
            + feature_df[time_col].astype(str),
            errors="coerce",
        )
    feature_df["hour_of_day"] = time_parsed.dt.hour
else:
    feature_df["hour_of_day"] = np.nan

feature_df["overnight_segment_flag"] = feature_df["hour_of_day"].isin(
    [22, 23, 0, 1, 2, 3, 4]
).astype(float)

# Segment features
feature_df["distance_to_next_station"] = (
    feature_df.groupby("journey_date")["distance_from_origin"].shift(-1)
    - feature_df["distance_from_origin"]
)

# Compute scheduled travel time using static schedule when available
next_arrival_min = feature_df.groupby("journey_date")["scheduled_arrival_min"].shift(-1)
feature_df["scheduled_travel_time_min"] = next_arrival_min - feature_df["scheduled_departure_min"]

feature_df["average_speed_expected_kmph"] = np.where(
    feature_df["scheduled_travel_time_min"].gt(0),
    feature_df["distance_to_next_station"]
    / (feature_df["scheduled_travel_time_min"] / 60.0),
    np.nan,
)
feature_df["average_speed_expected_kmph"] = feature_df[
    "average_speed_expected_kmph"
].replace([np.inf, -np.inf], np.nan)

# If average speed is missing, fall back to static speed on section
feature_df["average_speed_expected_kmph"] = feature_df[
    "average_speed_expected_kmph"
].fillna(feature_df["speed_on_section_kmph"])

# Historical statistical features (past-only expanding windows)
feature_df["next_station_code"] = feature_df.groupby("journey_date")["station_code"].shift(-1)
feature_df["segment_key"] = (
    feature_df["station_code"].astype(str)
    + "->"
    + feature_df["next_station_code"].astype(str)
 )

feature_df["avg_delay_at_station"] = feature_df.groupby("station_code")[
    "delay_departure_minutes"
].transform(lambda s: s.shift(1).expanding().mean())
feature_df["delay_variance_at_station"] = feature_df.groupby("station_code")[
    "delay_departure_minutes"
].transform(lambda s: s.shift(1).expanding().var())
feature_df["avg_delay_for_train"] = feature_df["delay_departure_minutes"].shift(1).expanding().mean()
feature_df["station_congestion_score"] = feature_df.groupby("station_code")[
    "delay_departure_minutes"
].transform(lambda s: s.shift(1).expanding().mean().abs())
feature_df["avg_delay_on_segment"] = feature_df.groupby("segment_key")[
    "target_next_delay"
].transform(lambda s: s.shift(1).expanding().mean())

# Feature Catalog (Train 18238)

| Feature | Description | Calculation / Source |
| --- | --- | --- |
| `current_delay` | Delay at the current station. | `delay_departure_minutes` at station $i$. |
| `previous_delay_1` | Delay at the previous station. | Shifted by 1 within each `journey_date`. |
| `previous_delay_2` | Delay two stations back. | Shifted by 2 within each `journey_date`. |
| `rolling_avg_last_3_delays` | Recent average delay trend. | Mean of the last 3 delays (past-only), per journey. |
| `rolling_std_last_3_delays` | Recent delay volatility. | Std dev of the last 3 delays (past-only), per journey. |
| `delay_trend_slope` | Recent trend direction. | Least-squares slope on last 3 past delays. |
| `station_index` | Station order along route. | Halt-only index from static route order. |
| `journey_progress_pct` | Progress along route. | Distance-from-source / max distance (or station index ratio if distance missing). |
| `day_of_week` | Weekday signal. | `journey_date.dt.dayofweek` (0=Mon). |
| `month` | Seasonality signal. | `journey_date.dt.month`. |
| `is_weekend` | Weekend flag. | 1 if day is Sat/Sun else 0. |
| `hour_of_day` | Time-of-day signal. | From actual/scheduled time column when available. |
| `overnight_segment_flag` | Late-night segment. | 1 if hour in {22,23,0,1,2,3,4}. |
| `distance_from_origin` | Distance from route start. | Static `distanceFromSourceKm` fallback if dataset distance is missing. |
| `distance_to_next_station` | Segment length. | Next station distance minus current distance within journey. |
| `scheduled_travel_time_min` | Scheduled segment travel time. | Next station scheduled arrival - current scheduled departure (static). |
| `halt_duration_min` | Scheduled halt at station. | Static `haltDurationMinutes`. |
| `average_speed_expected_kmph` | Expected segment speed. | Distance to next / scheduled travel time; fallback to static `speedOnSectionKmph`. |
| `avg_delay_at_station` | Typical delay at station. | Expanding mean of past delays per station (shifted). |
| `delay_variance_at_station` | Delay variability at station. | Expanding variance of past delays per station (shifted). |
| `avg_delay_for_train` | Overall train delay tendency. | Expanding mean of past delays across all rows (shifted). |
| `station_congestion_score` | Station delay pressure proxy. | Expanding mean of abs past delays per station (shifted). |
| `avg_delay_on_segment` | Typical delay for segment. | Expanding mean of past `target_next_delay` per segment (shifted). |

In [8]:
# Validation checks for feature engineering
expected_cols = [
    "current_delay",
    "previous_delay_1",
    "previous_delay_2",
    "rolling_avg_last_3_delays",
    "rolling_std_last_3_delays",
    "delay_trend_slope",
    "journey_progress_pct",
    "day_of_week",
    "month",
    "is_weekend",
    "hour_of_day",
    "overnight_segment_flag",
    "distance_from_origin",
    "distance_to_next_station",
    "scheduled_travel_time_min",
    "halt_duration_min",
    "average_speed_expected_kmph",
    "avg_delay_at_station",
    "avg_delay_on_segment",
    "avg_delay_for_train",
    "station_congestion_score",
    "delay_variance_at_station",
    "target_next_delay",
    "station_index",
    "next_station_index",
]

missing_cols = [c for c in expected_cols if c not in feature_df.columns]
print(f"Missing feature columns: {missing_cols}")

null_summary = feature_df[expected_cols].isna().mean().sort_values(ascending=False).head(12)
print("Top null rates:")
print(null_summary)

step = feature_df["next_station_index"] - feature_df["station_index"]
print("Next-station step distribution (top 5):")
print(step.value_counts().head(5))
print(f"Non-positive steps: {(step <= 0).sum()}")

# Spot-check first station rows per journey for expected lag nulls
first_station_rows = feature_df.groupby("journey_date").head(1)
print("First-station lag null rate:")
print(first_station_rows[["previous_delay_1", "previous_delay_2"]].isna().mean())

Missing feature columns: []
Top null rates:
previous_delay_1             0.075517
previous_delay_2             0.075517
delay_variance_at_station    0.071188
hour_of_day                  0.062771
current_delay                0.062771
station_congestion_score     0.057720
avg_delay_at_station         0.057720
rolling_std_last_3_delays    0.026215
delay_trend_slope            0.026215
distance_to_next_station     0.025734
avg_delay_on_segment         0.023810
journey_progress_pct         0.012867
dtype: float64
Next-station step distribution (top 5):
1    8316
Name: count, dtype: int64
Non-positive steps: 0
First-station lag null rate:
previous_delay_1    1.0
previous_delay_2    1.0
dtype: float64


# Phase 2 - Baselines (Train 18238)

This section builds simple baselines using a time-based split on journey dates to avoid leakage.

In [9]:
# Time-based split by journey_date
unique_dates = sorted(feature_df["journey_date"].unique())
if len(unique_dates) < 5:
    raise ValueError("Not enough journeys for a time-based split.")

split_idx = int(len(unique_dates) * 0.8)
train_dates = set(unique_dates[:split_idx])
test_dates = set(unique_dates[split_idx:])

train_set = feature_df[feature_df["journey_date"].isin(train_dates)].copy()
test_set = feature_df[feature_df["journey_date"].isin(test_dates)].copy()

print(f"Train journeys: {len(train_dates)} | Test journeys: {len(test_dates)}")
print(f"Train rows: {len(train_set):,} | Test rows: {len(test_set):,}")

feature_cols = [
    "current_delay",
    "previous_delay_1",
    "previous_delay_2",
    "rolling_avg_last_3_delays",
    "rolling_std_last_3_delays",
    "delay_trend_slope",
    "journey_progress_pct",
    "day_of_week",
    "month",
    "is_weekend",
    "hour_of_day",
    "overnight_segment_flag",
    "station_index",
    "distance_from_origin",
    "distance_to_next_station",
    "scheduled_travel_time_min",
    "halt_duration_min",
    "average_speed_expected_kmph",
    "avg_delay_at_station",
    "avg_delay_on_segment",
    "avg_delay_for_train",
    "station_congestion_score",
    "delay_variance_at_station",
]

usable_feature_cols = [c for c in feature_cols if train_set[c].notna().any()]
print(f"Features used: {len(usable_feature_cols)}")

# Prepare model sets and impute missing feature values with train medians
train_model = train_set.dropna(subset=["target_next_delay"]).copy()
test_model = test_set.dropna(subset=["target_next_delay"]).copy()

median_values = train_model[usable_feature_cols].median(numeric_only=True).fillna(0)
train_model[usable_feature_cols] = train_model[usable_feature_cols].fillna(median_values)
test_model[usable_feature_cols] = test_model[usable_feature_cols].fillna(median_values)

print(f"Model-ready train rows: {len(train_model):,}")
print(f"Model-ready test rows: {len(test_model):,}")

# Baseline predictors
baseline_predictions = {}

# 1) Persistence baseline (predict current delay for next station)
baseline_predictions["persistence"] = test_model["current_delay"].values

# 2) Rolling mean baseline
baseline_predictions["rolling_mean_3"] = test_model["rolling_avg_last_3_delays"].values

# 3) Simple linear regression (numpy least squares with intercept)
X_train = train_model[usable_feature_cols].values
X_test = test_model[usable_feature_cols].values

y_train = train_model["target_next_delay"].values
y_test = test_model["target_next_delay"].values

# Add intercept column
X_train_i = np.column_stack([np.ones(len(X_train)), X_train])
X_test_i = np.column_stack([np.ones(len(X_test)), X_test])

beta, *_ = np.linalg.lstsq(X_train_i, y_train, rcond=None)
lin_pred = X_test_i @ beta
baseline_predictions["linear_lstsq"] = lin_pred

# 4) Ridge regression (closed-form)
alpha = 1.0
XtX = X_train_i.T @ X_train_i
ridge_beta = np.linalg.solve(XtX + alpha * np.eye(XtX.shape[0]), X_train_i.T @ y_train)
ridge_pred = X_test_i @ ridge_beta
baseline_predictions["ridge_alpha_1"] = ridge_pred

# Evaluation metrics

def evaluate_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return {"mae": mae, "rmse": rmse}

results = []
for name, preds in baseline_predictions.items():
    metrics = evaluate_metrics(y_test, preds)
    results.append({"model": name, **metrics})

results_df = pd.DataFrame(results).sort_values("mae")
results_df.reset_index(drop=True, inplace=True)

print("Baseline results (sorted by MAE):")
display(results_df)

Train journeys: 85 | Test journeys: 22
Train rows: 6,597 | Test rows: 1,719
Features used: 23
Model-ready train rows: 6,597
Model-ready test rows: 1,719
Baseline results (sorted by MAE):


,model,mae,rmse
0,linear_lstsq,7.160496,10.276799
1,ridge_alpha_1,7.161159,10.274615
2,persistence,8.374636,14.190469
3,rolling_mean_3,12.256738,17.099556


# Phase 2.5 — Baseline Evaluation Visualizations

In [10]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os

os.makedirs("../outputs/plots", exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Baseline Model Evaluation — Train 18238", fontsize=14, fontweight="bold")

model_names   = list(baseline_predictions.keys())
colors        = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
sample_idx    = min(200, len(y_test))
x_range       = range(sample_idx)

# Actual vs Predicted (persistence)
ax = axes[0, 0]
ax.plot(x_range, y_test[:sample_idx], label="Actual", color="black", linewidth=1.2)
ax.plot(x_range, baseline_predictions["persistence"][:sample_idx],
        label="Persistence", color=colors[0], linewidth=1, linestyle="--")
ax.set_title("Actual vs Persistence"); ax.set_xlabel("Sample"); ax.set_ylabel("Delay (min)")
ax.legend(); ax.grid(alpha=0.3)

# Actual vs Rolling Mean
ax = axes[0, 1]
ax.plot(x_range, y_test[:sample_idx], label="Actual", color="black", linewidth=1.2)
ax.plot(x_range, baseline_predictions["rolling_mean_3"][:sample_idx],
        label="Rolling Mean 3", color=colors[1], linewidth=1, linestyle="--")
ax.set_title("Actual vs Rolling Mean 3"); ax.set_xlabel("Sample"); ax.set_ylabel("Delay (min)")
ax.legend(); ax.grid(alpha=0.3)

# Residuals for Ridge
ax = axes[1, 0]
residuals = y_test - baseline_predictions["ridge_alpha_1"]
ax.scatter(range(len(residuals)), residuals, alpha=0.4, s=8, color=colors[3])
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Ridge Residuals"); ax.set_xlabel("Sample"); ax.set_ylabel("Residual (min)")
ax.grid(alpha=0.3)

# Error histogram (linear vs ridge)
ax = axes[1, 1]
for name, col in [("linear_lstsq", colors[2]), ("ridge_alpha_1", colors[3])]:
    errors = y_test - baseline_predictions[name]
    ax.hist(errors, bins=40, alpha=0.5, label=name, color=col)
ax.set_title("Prediction Error Histogram"); ax.set_xlabel("Error (min)"); ax.set_ylabel("Count")
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../outputs/plots/baseline_evaluation.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: ../outputs/plots/baseline_evaluation.png")


Saved: ../outputs/plots/baseline_evaluation.png


C:\Users\arjun\AppData\Local\Temp\ipykernel_11080\3707988777.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Phase 3 — XGBoost Model (Train 18238)

The primary production model. We train XGBoost, tune hyperparameters with Optuna, analyse SHAP feature importance, implement recursive multi-station forecasting, and serialize the model for API use.

In [11]:
# Step 3.1-3.2 — Initial XGBoost Training
try:
    import xgboost as xgb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost", "-q"])
    import xgboost as xgb

from sklearn.preprocessing import StandardScaler
import pickle, json as _json

print(f"XGBoost version: {xgb.__version__}")

# Reuse train_model / test_model and usable_feature_cols from Phase 2
X_tr = train_model[usable_feature_cols].values
y_tr = train_model["target_next_delay"].values
X_te = test_model[usable_feature_cols].values
y_te = test_model["target_next_delay"].values

# ------ validation split (last 10 % of train dates for early stopping) ------
val_split = int(len(X_tr) * 0.9)
X_tr_fit, X_val = X_tr[:val_split], X_tr[val_split:]
y_tr_fit, y_val = y_tr[:val_split], y_tr[val_split:]

dtrain = xgb.DMatrix(X_tr_fit, label=y_tr_fit, feature_names=usable_feature_cols)
dval   = xgb.DMatrix(X_val,   label=y_val,   feature_names=usable_feature_cols)
dtest  = xgb.DMatrix(X_te,               feature_names=usable_feature_cols)

params_init = {
    "objective": "reg:squarederror",
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 5,
    "seed": 42,
    "eval_metric": "mae",
}

xgb_init = xgb.train(
    params_init,
    dtrain,
    num_boost_round=300,
    evals=[(dtrain, "train"), (dval, "val")],
    early_stopping_rounds=30,
    verbose_eval=False,
)

init_preds = xgb_init.predict(dtest)
init_mae   = np.mean(np.abs(y_te - init_preds))
init_rmse  = np.sqrt(np.mean((y_te - init_preds) ** 2))
print(f"Initial XGBoost — MAE: {init_mae:.3f} | RMSE: {init_rmse:.3f}")
print(f"Best iteration: {xgb_init.best_iteration}")


XGBoost version: 3.2.0
Initial XGBoost — MAE: 5.643 | RMSE: 8.728
Best iteration: 123


In [12]:
# Step 3.3 — Hyperparameter Tuning with Optuna
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna", "-q"])
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        "objective": "reg:squarederror",
        "eval_metric": "mae",
        "seed": 42,
        "max_depth":          trial.suggest_int("max_depth", 3, 8),
        "learning_rate":      trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "min_child_weight":   trial.suggest_int("min_child_weight", 3, 20),
        "subsample":          trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":   trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma":              trial.suggest_float("gamma", 0, 5),
        "reg_alpha":          trial.suggest_float("reg_alpha", 0, 2),
        "reg_lambda":         trial.suggest_float("reg_lambda", 0.5, 5),
    }
    model = xgb.train(
        params, dtrain,
        num_boost_round=500,
        evals=[(dval, "val")],
        early_stopping_rounds=30,
        verbose_eval=False,
    )
    return model.best_score   # val MAE

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=False)

print(f"Best trial MAE: {study.best_value:.4f}")
print("Best params:", study.best_params)


d:\Code\Python Projects\railway\rail_data\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Best trial MAE: 5.1016
Best params: {'max_depth': 8, 'learning_rate': 0.03918847249405732, 'min_child_weight': 15, 'subsample': 0.7118615413711042, 'colsample_bytree': 0.781869127978545, 'gamma': 1.5349265993338728, 'reg_alpha': 1.8022986054914478, 'reg_lambda': 4.253836549514104}


In [13]:
# Step 3.3 — Train Final XGBoost with Best Params
best_params = study.best_params
best_params.update({"objective": "reg:squarederror", "eval_metric": "mae", "seed": 42})

# Use full training set (no val split) for final model
dtrain_full = xgb.DMatrix(X_tr, label=y_tr, feature_names=usable_feature_cols)
n_rounds = study.best_trial.user_attrs.get("n_rounds",
              xgb.train(best_params, dtrain, num_boost_round=500,
                        evals=[(dval, "val")], early_stopping_rounds=30,
                        verbose_eval=False).best_iteration + 1)

xgb_best = xgb.train(
    best_params, dtrain_full,
    num_boost_round=int(n_rounds * 1.1),   # slight boost with full data
    verbose_eval=False,
)

best_preds = xgb_best.predict(dtest)
best_mae   = np.mean(np.abs(y_te - best_preds))
best_rmse  = np.sqrt(np.mean((y_te - best_preds) ** 2))
print(f"Tuned XGBoost — MAE: {best_mae:.3f} | RMSE: {best_rmse:.3f}")


Tuned XGBoost — MAE: 5.532 | RMSE: 8.234


In [14]:
# Step 3.4 — Feature Importance & SHAP Analysis
try:
    import shap
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap", "-q"])
    import shap

# XGBoost native gain importance
gain_imp = xgb_best.get_score(importance_type="gain")
imp_df = pd.DataFrame(sorted(gain_imp.items(), key=lambda x: -x[1]),
                      columns=["feature", "gain"])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("XGBoost Feature Importance — Train 18238", fontsize=13, fontweight="bold")

ax = axes[0]
top_n = imp_df.head(15)
ax.barh(top_n["feature"][::-1], top_n["gain"][::-1], color="#4C72B0")
ax.set_title("Gain Importance (Top 15)"); ax.set_xlabel("Gain")
ax.grid(axis="x", alpha=0.3)

# SHAP values
explainer = shap.TreeExplainer(xgb_best)
shap_vals  = explainer.shap_values(dtest)
shap_mean  = np.abs(shap_vals).mean(axis=0)
shap_df    = pd.DataFrame({"feature": usable_feature_cols, "mean_abs_shap": shap_mean})
shap_df    = shap_df.sort_values("mean_abs_shap", ascending=False).head(15)

ax = axes[1]
ax.barh(shap_df["feature"][::-1], shap_df["mean_abs_shap"][::-1], color="#DD8452")
ax.set_title("SHAP Mean |shap| (Top 15)"); ax.set_xlabel("Mean |SHAP Value|")
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig("../outputs/plots/xgb_feature_importance.png", dpi=120, bbox_inches="tight")
plt.show()
print("Top 5 features by SHAP:")
print(shap_df.head(5).to_string(index=False))


Top 5 features by SHAP:
                    feature  mean_abs_shap
              current_delay      12.326658
  rolling_avg_last_3_delays       4.419219
           previous_delay_1       2.838585
       avg_delay_on_segment       2.441083
average_speed_expected_kmph       2.104140


C:\Users\arjun\AppData\Local\Temp\ipykernel_11080\2873933297.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# Step 3.5 — Recursive Multi-Station Prediction
# Given the current state at station i, predict all future stations in a journey.

def recursive_forecast(model, journey_df: pd.DataFrame,
                       feature_cols: list, start_idx: int = 0) -> pd.DataFrame:
    """
    Forecast delays for all stations from start_idx onward within a single journey.
    The predicted next-delay is fed back as 'current_delay' for the following step.
    Returns a DataFrame with station_code, actual, predicted columns.
    """
    journey_df = journey_df.sort_values("station_index").reset_index(drop=True)
    rows = journey_df.copy()

    results = []
    rolling_window = []

    for i in range(start_idx, len(rows) - 1):
        row = rows.iloc[i].copy()

        # Fill rolling features from our maintained window
        if len(rolling_window) >= 1:
            row["previous_delay_1"] = rolling_window[-1]
        if len(rolling_window) >= 2:
            row["previous_delay_2"] = rolling_window[-2]
        if len(rolling_window) >= 1:
            w = rolling_window[-3:]
            row["rolling_avg_last_3_delays"] = np.mean(w)
            row["rolling_std_last_3_delays"] = np.std(w) if len(w) > 1 else 0.0
        if len(rolling_window) >= 2:
            y = np.array(rolling_window[-3:], dtype=float)
            x = np.arange(len(y), dtype=float)
            if len(y) >= 2 and np.ptp(x) > 0:
                row["delay_trend_slope"] = np.polyfit(x, y, 1)[0]

        feat_vals = row[feature_cols].values.reshape(1, -1)
        dm = xgb.DMatrix(feat_vals, feature_names=feature_cols)
        pred = float(model.predict(dm)[0])

        actual_next = rows.iloc[i + 1]["delay_departure_minutes"]
        results.append({
            "from_station": row["station_code"],
            "to_station":   rows.iloc[i + 1]["station_code"],
            "station_index": int(rows.iloc[i + 1]["station_index"]),
            "actual_next":  actual_next,
            "predicted_next": pred,
        })

        rolling_window.append(float(row["current_delay"]))
        # Update current_delay for next step with our prediction
        rows.at[i + 1, "current_delay"] = pred

    return pd.DataFrame(results)

# Demo: pick the first test journey and forecast recursively
sample_date = sorted(test_dates)[0]
sample_journey = test_model[test_model["journey_date"] == sample_date].copy()
sample_journey = sample_journey.sort_values("station_index").reset_index(drop=True)

if len(sample_journey) >= 4:
    forecast_df = recursive_forecast(xgb_best, sample_journey, usable_feature_cols, start_idx=0)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(forecast_df["station_index"], forecast_df["actual_next"],
            marker="o", label="Actual", color="black", linewidth=1.5)
    ax.plot(forecast_df["station_index"], forecast_df["predicted_next"],
            marker="s", label="XGBoost Recursive Forecast", color="#4C72B0",
            linewidth=1.5, linestyle="--")
    ax.set_title(f"Recursive Multi-Station Forecast — Journey {sample_date}")
    ax.set_xlabel("Station Index"); ax.set_ylabel("Delay (min)")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("../outputs/plots/xgb_recursive_forecast.png", dpi=120, bbox_inches="tight")
    plt.show()

    rec_mae  = np.mean(np.abs(forecast_df["actual_next"] - forecast_df["predicted_next"]))
    rec_rmse = np.sqrt(np.mean((forecast_df["actual_next"] - forecast_df["predicted_next"]) ** 2))
    print(f"Recursive forecast MAE: {rec_mae:.3f} | RMSE: {rec_rmse:.3f}")
    display(forecast_df)
else:
    print(f"Sample journey too short ({len(sample_journey)} rows), skipping demo.")


Recursive forecast MAE: 11.273 | RMSE: 15.564


C:\Users\arjun\AppData\Local\Temp\ipykernel_11080\3681918065.py:73: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,from_station,to_station,station_index,actual_next,predicted_next
0,ASR,JNL,1,12.0,13.468005
1,JNL,BEAS,2,24.0,17.917885
2,BEAS,JUC,3,13.0,12.066993
3,JUC,JRC,4,2.0,5.403928
4,JRC,PGW,5,9.0,9.016809
...,...,...,...,...,...
72,DGG,DURG,77,NaN,32.711426
73,DURG,BPHB,78,4.0,23.839678
74,BPHB,R,79,29.0,34.602673
75,R,HN,81,NaN,45.694153


In [16]:
# Step 3.6 — Prediction Confidence Estimation (Residual Std Approach)

# Compute per-station residual std from training predictions
dtrain_pred_dm  = xgb.DMatrix(X_tr, feature_names=usable_feature_cols)
train_residuals = y_tr - xgb_best.predict(dtrain_pred_dm)

# Per-station-index residual std (for confidence band calibration)
train_model_copy = train_model.copy()
train_model_copy["residual"] = train_residuals
station_residual_std = (
    train_model_copy.groupby("station_index")["residual"]
    .std()
    .fillna(train_residuals.std())
)

# Apply to test predictions
test_model_copy = test_model.copy()
test_model_copy["predicted"]     = best_preds
test_model_copy["residual_std"]  = test_model_copy["station_index"].map(station_residual_std).fillna(train_residuals.std())
test_model_copy["lower_bound"]   = test_model_copy["predicted"] - 1.5 * test_model_copy["residual_std"]
test_model_copy["upper_bound"]   = test_model_copy["predicted"] + 1.5 * test_model_copy["residual_std"]

# Coverage check (what % of actuals fall within confidence band)
coverage = (
    (test_model_copy["target_next_delay"] >= test_model_copy["lower_bound"]) &
    (test_model_copy["target_next_delay"] <= test_model_copy["upper_bound"])
).mean()

print(f"Confidence band coverage (~90% target): {coverage:.1%}")

# Plot confidence bands for a slice of test data
sample = test_model_copy.head(80).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(sample.index, sample["target_next_delay"], label="Actual", color="black", linewidth=1.4)
ax.plot(sample.index, sample["predicted"], label="XGBoost Prediction", color="#4C72B0",
        linewidth=1.4, linestyle="--")
ax.fill_between(sample.index, sample["lower_bound"], sample["upper_bound"],
                alpha=0.25, color="#4C72B0", label="±1.5σ Confidence")
ax.set_title("XGBoost Predictions with Confidence Bands")
ax.set_xlabel("Sample"); ax.set_ylabel("Delay (min)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../outputs/plots/xgb_confidence_bands.png", dpi=120, bbox_inches="tight")
plt.show()


Confidence band coverage (~90% target): 85.2%


C:\Users\arjun\AppData\Local\Temp\ipykernel_11080\1175819801.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
# Step 3.7 — Error Analysis

error_df = test_model.copy()
error_df["predicted"] = best_preds
error_df["abs_error"] = np.abs(error_df["target_next_delay"] - error_df["predicted"])
error_df["error"]     = error_df["target_next_delay"] - error_df["predicted"]

print("=== Worst 10 Predictions ===")
display(error_df.nlargest(10, "abs_error")[["journey_date", "station_code",
         "target_next_delay", "predicted", "abs_error"]])

# Per-station MAE
station_mae = error_df.groupby("station_code")["abs_error"].mean().sort_values(ascending=False)
print("\n=== Highest MAE Stations (top 10) ===")
print(station_mae.head(10))

# Night-time vs daytime performance
if "hour_of_day" in error_df.columns and error_df["hour_of_day"].notna().any():
    night_mask = error_df["overnight_segment_flag"] == 1
    print(f"\nNight MAE  : {error_df.loc[night_mask, 'abs_error'].mean():.3f}")
    print(f"Day   MAE  : {error_df.loc[~night_mask, 'abs_error'].mean():.3f}")

# High-delay scenario (actual > 30 min)
high_delay = error_df[error_df["target_next_delay"] > 30]
if len(high_delay) > 0:
    print(f"\nHigh-delay MAE (actual>30 min): {high_delay['abs_error'].mean():.3f} over {len(high_delay)} samples")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("XGBoost Error Analysis", fontsize=13, fontweight="bold")

# Error histogram
axes[0].hist(error_df["error"], bins=50, color="#C44E52", alpha=0.75, edgecolor="white")
axes[0].axvline(0, color="black", linewidth=1.2)
axes[0].set_title("Error Distribution"); axes[0].set_xlabel("Error (min)")
axes[0].grid(alpha=0.3)

# Actual vs Predicted scatter
axes[1].scatter(error_df["target_next_delay"], error_df["predicted"],
                alpha=0.3, s=8, color="#4C72B0")
lo = min(error_df["target_next_delay"].min(), error_df["predicted"].min())
hi = max(error_df["target_next_delay"].max(), error_df["predicted"].max())
axes[1].plot([lo, hi], [lo, hi], "r--", linewidth=1.2, label="Perfect")
axes[1].set_title("Actual vs Predicted"); axes[1].set_xlabel("Actual"); axes[1].set_ylabel("Predicted")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../outputs/plots/xgb_error_analysis.png", dpi=120, bbox_inches="tight")
plt.show()


=== Worst 10 Predictions ===


,journey_date,station_code,target_next_delay,predicted,abs_error
7982,2026-05-20,ET,83.0,19.966335,63.033665
7339,2026-05-12,DHO,78.0,31.213160,46.786840
7728,2026-05-17,DHO,81.0,35.735001,45.264999
6981,2026-05-07,SKS,82.0,39.764553,42.235447
7963,2026-05-20,DHO,80.0,41.084091,38.915909
8273,2026-05-24,MTJ,113.0,74.880730,38.119270
7573,2026-05-15,DHO,72.0,34.788479,37.211521
7993,2026-05-20,TRO,64.0,28.124550,35.875450
6743,2026-05-04,TRO,68.0,32.154377,35.845623
6888,2026-05-06,ET,50.0,14.225190,35.774810



=== Highest MAE Stations (top 10) ===
station_code
DHO    25.706462
TRO    15.654040
MTJ    15.647238
RAA    14.548044
SKS    12.153456
ET     11.059098
UMB    10.042760
R       9.906417
GWL     9.588936
UBC     9.017334
Name: abs_error, dtype: float64

Night MAE  : 5.485
Day   MAE  : 6.247

High-delay MAE (actual>30 min): 9.795 over 376 samples


C:\Users\arjun\AppData\Local\Temp\ipykernel_11080\1500983693.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
# Step 3.8 — Model Serialization
import pickle, os, json as _json

os.makedirs("../outputs/models", exist_ok=True)

# Save XGBoost model (native format + pickle)
xgb_best.save_model("../outputs/models/xgb_train18238.json")

with open("../outputs/models/xgb_train18238.pkl", "wb") as f:
    pickle.dump(xgb_best, f)

# Save feature list
with open("../outputs/models/feature_columns.json", "w") as f:
    _json.dump({"features": usable_feature_cols}, f, indent=2)

# Save imputation medians (scaler proxy)
with open("../outputs/models/median_imputer.pkl", "wb") as f:
    pickle.dump(median_values.to_dict(), f)

# Save station residual std for confidence intervals
with open("../outputs/models/station_residual_std.pkl", "wb") as f:
    pickle.dump(station_residual_std.to_dict(), f)

print("Model artifacts saved:")
for fn in os.listdir("../outputs/models"):
    fpath = os.path.join("../outputs/models", fn)
    print(f"  {fn:40s}  {os.path.getsize(fpath):>8,} bytes")


Model artifacts saved:
  feature_columns.json                           657 bytes
  lstm_train18238.pt                         228,741 bytes
  median_imputer.pkl                             717 bytes
  scaler_seq.pkl                                 968 bytes
  station_residual_std.pkl                       896 bytes
  tcn_train18238.pt                          225,365 bytes
  xgb_train18238.json                        880,980 bytes
  xgb_train18238.pkl                         635,637 bytes


In [19]:
# Step 3.9 — Flask Inference API (inline definition, run separately)

FLASK_API_CODE = '''
from flask import Flask, request, jsonify
import xgboost as xgb
import numpy as np
import pickle, json

app = Flask(__name__)

MODEL_PATH   = "../outputs/models/xgb_train18238.json"
FEAT_PATH    = "../outputs/models/feature_columns.json"
MEDIAN_PATH  = "../outputs/models/median_imputer.pkl"
STD_PATH     = "../outputs/models/station_residual_std.pkl"

model = xgb.Booster(); model.load_model(MODEL_PATH)
with open(FEAT_PATH) as f:  feature_cols = json.load(f)["features"]
with open(MEDIAN_PATH, "rb") as f:  medians = pickle.load(f)
with open(STD_PATH, "rb") as f:  station_std = pickle.load(f)

@app.route("/predict-delay", methods=["POST"])
def predict_delay():
    body = request.get_json(force=True)
    prev = body.get("previous_delays", [])
    row = {c: medians.get(c, 0) for c in feature_cols}
    row["current_delay"]            = body.get("current_delay", 0)
    row["previous_delay_1"]         = prev[-1] if len(prev) >= 1 else row["current_delay"]
    row["previous_delay_2"]         = prev[-2] if len(prev) >= 2 else row["previous_delay_1"]
    row["station_index"]            = body.get("station_index", 0)
    row["day_of_week"]              = body.get("day_of_week", 0)
    row["month"]                    = body.get("month", 1)
    row["hour_of_day"]              = body.get("hour_of_day", 12)
    row["rolling_avg_last_3_delays"] = np.mean([row["current_delay"],
                                                  row["previous_delay_1"],
                                                  row["previous_delay_2"]])
    feat = np.array([[row[c] for c in feature_cols]])
    dm   = xgb.DMatrix(feat, feature_names=feature_cols)
    pred = float(model.predict(dm)[0])
    s_idx = int(body.get("station_index", 0))
    std  = station_std.get(s_idx, np.std(list(station_std.values())))
    return jsonify({
        "next_station_delay": round(pred, 2),
        "confidence_range": [round(pred - 1.5*std, 2), round(pred + 1.5*std, 2)]
    })

if __name__ == "__main__":
    app.run(port=5001, debug=False)
'''

os.makedirs("../outputs", exist_ok=True)
with open("../outputs/inference_api.py", "w") as f:
    f.write(FLASK_API_CODE)

print("Inference API written to: ../outputs/inference_api.py")
print()
print("Sample request:")
print('''  POST /predict-delay
  {
    "train_no": 18238,
    "current_station": "BPL",
    "current_delay": 18,
    "previous_delays": [12, 10],
    "station_index": 5,
    "hour_of_day": 14,
    "day_of_week": 2,
    "month": 5
  }''')


Inference API written to: ../outputs/inference_api.py

Sample request:
  POST /predict-delay
  {
    "train_no": 18238,
    "current_station": "BPL",
    "current_delay": 18,
    "previous_delays": [12, 10],
    "station_index": 5,
    "hour_of_day": 14,
    "day_of_week": 2,
    "month": 5
  }


In [20]:
# Phase 3 — Comparative Results Summary

xgb_mae  = float(np.mean(np.abs(y_te - best_preds)))
xgb_rmse = float(np.sqrt(np.mean((y_te - best_preds) ** 2)))

all_results = results_df.copy()   # baselines from Phase 2

xgb_row = pd.DataFrame([{"model": "xgboost_tuned", "mae": xgb_mae, "rmse": xgb_rmse}])
all_results = pd.concat([all_results, xgb_row], ignore_index=True).sort_values("mae")

print("=== Model Comparison (Phases 2 & 3) ===")
display(all_results.style.format({"mae": "{:.3f}", "rmse": "{:.3f}"}))


=== Model Comparison (Phases 2 & 3) ===


,model,mae,rmse
4,xgboost_tuned,5.532,8.234
0,linear_lstsq,7.160,10.277
1,ridge_alpha_1,7.161,10.275
2,persistence,8.375,14.190
3,rolling_mean_3,12.257,17.100


# Phase 4 — Advanced Sequence Models (TCN & LSTM)

Research-oriented models for comparison. We use PyTorch for both architectures, with time-based splitting before sequence windowing.

In [21]:
# Step 4.1-4.2 — Sequence Dataset Generation
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "--index-url",
                           "https://download.pytorch.org/whl/cpu", "-q"])
    import torch, torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler

print(f"PyTorch version: {torch.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

SEQ_LEN  = 4   # use last SEQ_LEN stations to predict the next
N_FEATS  = len(usable_feature_cols)

def build_sequences(df, feature_cols, target_col="target_next_delay", seq_len=SEQ_LEN):
    """Build (X_seq, y) arrays grouped by journey_date, respecting temporal order."""
    X_seqs, ys = [], []
    for _, journey in df.groupby("journey_date"):
        journey = journey.sort_values("station_index")
        feats = journey[feature_cols].values.astype(np.float32)
        tgts  = journey[target_col].values.astype(np.float32)
        for i in range(seq_len, len(feats)):
            X_seqs.append(feats[i - seq_len:i])
            ys.append(tgts[i - 1])           # predict target at step i-1 (next after seq end)
    if not X_seqs:
        return np.empty((0, seq_len, len(feature_cols)), np.float32), np.empty(0, np.float32)
    return np.stack(X_seqs), np.array(ys, dtype=np.float32)

# Scale features with train-fit scaler
scaler_seq = StandardScaler()
train_model_scaled = train_model.copy()
test_model_scaled  = test_model.copy()
train_model_scaled[usable_feature_cols] = scaler_seq.fit_transform(
    train_model[usable_feature_cols].values)
test_model_scaled[usable_feature_cols]  = scaler_seq.transform(
    test_model[usable_feature_cols].values)

X_seq_tr, y_seq_tr = build_sequences(train_model_scaled, usable_feature_cols)
X_seq_te, y_seq_te = build_sequences(test_model_scaled,  usable_feature_cols)

print(f"Train sequences: {X_seq_tr.shape}  |  Test sequences: {X_seq_te.shape}")

# Torch datasets
def make_loader(X, y, batch_size=64, shuffle=False):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

BATCH = 64
train_loader = make_loader(X_seq_tr, y_seq_tr, BATCH, shuffle=False)
test_loader  = make_loader(X_seq_te, y_seq_te, BATCH, shuffle=False)


PyTorch version: 2.12.0+cpu
Device: cpu
Train sequences: (6257, 4, 23)  |  Test sequences: (1631, 4, 23)


In [22]:
# Step 4.3 — TCN Model Architecture

class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__(); self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.2):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size, padding=pad, dilation=dilation),
            Chomp1d(pad),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_ch, out_ch, kernel_size, padding=pad, dilation=dilation),
            Chomp1d(pad),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TCN(nn.Module):
    def __init__(self, n_features, n_channels=(64, 64, 32), kernel_size=3, dropout=0.2):
        super().__init__()
        layers = []
        in_ch = n_features
        for i, out_ch in enumerate(n_channels):
            layers.append(TCNBlock(in_ch, out_ch, kernel_size, dilation=2**i, dropout=dropout))
            in_ch = out_ch
        self.network = nn.Sequential(*layers)
        self.head    = nn.Linear(in_ch, 1)

    def forward(self, x):
        # x: (batch, seq_len, features) -> (batch, features, seq_len)
        x = x.permute(0, 2, 1)
        y = self.network(x)
        return self.head(y[:, :, -1]).squeeze(-1)

tcn_model = TCN(n_features=N_FEATS).to(DEVICE)
print(tcn_model)
total_params = sum(p.numel() for p in tcn_model.parameters())
print(f"TCN total parameters: {total_params:,}")


TCN(
  (network): Sequential(
    (0): TCNBlock(
      (net): Sequential(
        (0): Conv1d(23, 64, kernel_size=(3,), stride=(1,), padding=(2,))
        (1): Chomp1d()
        (2): ReLU()
        (3): Dropout(p=0.2, inplace=False)
        (4): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(2,))
        (5): Chomp1d()
        (6): ReLU()
        (7): Dropout(p=0.2, inplace=False)
      )
      (downsample): Conv1d(23, 64, kernel_size=(1,), stride=(1,))
      (relu): ReLU()
    )
    (1): TCNBlock(
      (net): Sequential(
        (0): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(4,), dilation=(2,))
        (1): Chomp1d()
        (2): ReLU()
        (3): Dropout(p=0.2, inplace=False)
        (4): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(4,), dilation=(2,))
        (5): Chomp1d()
        (6): ReLU()
        (7): Dropout(p=0.2, inplace=False)
      )
      (relu): ReLU()
    )
    (2): TCNBlock(
      (net): Sequential(
        (0): Conv1d(64, 32, kernel_s

In [23]:
# Step 4.3 — Train TCN

def train_epoch(model, loader, optimizer, criterion):
    model.train(); total_loss = 0
    for Xb, yb in loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        pred = model(Xb)
        loss = criterion(pred, yb)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * len(yb)
    return total_loss / len(loader.dataset)

def eval_model(model, loader):
    model.eval(); preds, actuals = [], []
    with torch.no_grad():
        for Xb, yb in loader:
            p = model(Xb.to(DEVICE)).cpu().numpy()
            preds.extend(p); actuals.extend(yb.numpy())
    preds, actuals = np.array(preds), np.array(actuals)
    mae  = np.mean(np.abs(actuals - preds))
    rmse = np.sqrt(np.mean((actuals - preds) ** 2))
    return preds, actuals, mae, rmse

N_EPOCHS = 60
criterion = nn.L1Loss()
opt_tcn   = torch.optim.Adam(tcn_model.parameters(), lr=1e-3, weight_decay=1e-4)
sched_tcn = torch.optim.lr_scheduler.StepLR(opt_tcn, step_size=20, gamma=0.5)

tcn_train_losses = []
best_tcn_mae, best_tcn_state = float("inf"), None

for epoch in range(1, N_EPOCHS + 1):
    loss = train_epoch(tcn_model, train_loader, opt_tcn, criterion)
    tcn_train_losses.append(loss)
    _, _, val_mae, _ = eval_model(tcn_model, test_loader)
    if val_mae < best_tcn_mae:
        best_tcn_mae = val_mae
        best_tcn_state = {k: v.clone() for k, v in tcn_model.state_dict().items()}
    sched_tcn.step()
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Train L1: {loss:.4f} | Test MAE: {val_mae:.4f}")

tcn_model.load_state_dict(best_tcn_state)
tcn_preds, tcn_actuals, tcn_mae, tcn_rmse = eval_model(tcn_model, test_loader)
print(f"\nTCN — MAE: {tcn_mae:.3f} | RMSE: {tcn_rmse:.3f}")


Epoch  10 | Train L1: 8.6866 | Test MAE: 7.2339
Epoch  20 | Train L1: 6.5544 | Test MAE: 5.8502
Epoch  30 | Train L1: 5.9851 | Test MAE: 5.1229
Epoch  40 | Train L1: 5.7319 | Test MAE: 4.9508
Epoch  50 | Train L1: 5.3890 | Test MAE: 4.8300
Epoch  60 | Train L1: 5.1252 | Test MAE: 4.7889

TCN — MAE: 4.778 | RMSE: 8.105


In [24]:
# Step 4.4 — LSTM Model Architecture & Training

class StackedLSTM(nn.Module):
    def __init__(self, n_features, hidden=64, n_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, num_layers=n_layers,
                            batch_first=True, dropout=dropout if n_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.head    = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(self.dropout(out[:, -1, :])).squeeze(-1)

lstm_model = StackedLSTM(n_features=N_FEATS).to(DEVICE)
total_params_lstm = sum(p.numel() for p in lstm_model.parameters())
print(f"LSTM total parameters: {total_params_lstm:,}")

opt_lstm   = torch.optim.Adam(lstm_model.parameters(), lr=1e-3, weight_decay=1e-4)
sched_lstm = torch.optim.lr_scheduler.StepLR(opt_lstm, step_size=20, gamma=0.5)

lstm_train_losses = []
best_lstm_mae, best_lstm_state = float("inf"), None

for epoch in range(1, N_EPOCHS + 1):
    loss = train_epoch(lstm_model, train_loader, opt_lstm, criterion)
    lstm_train_losses.append(loss)
    _, _, val_mae, _ = eval_model(lstm_model, test_loader)
    if val_mae < best_lstm_mae:
        best_lstm_mae = val_mae
        best_lstm_state = {k: v.clone() for k, v in lstm_model.state_dict().items()}
    sched_lstm.step()
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Train L1: {loss:.4f} | Test MAE: {val_mae:.4f}")

lstm_model.load_state_dict(best_lstm_state)
lstm_preds, lstm_actuals, lstm_mae, lstm_rmse = eval_model(lstm_model, test_loader)
print(f"\nLSTM — MAE: {lstm_mae:.3f} | RMSE: {lstm_rmse:.3f}")


LSTM total parameters: 56,129
Epoch  10 | Train L1: 17.3572 | Test MAE: 7.3920
Epoch  20 | Train L1: 12.1800 | Test MAE: 5.5191
Epoch  30 | Train L1: 10.2837 | Test MAE: 5.1811
Epoch  40 | Train L1: 9.1209 | Test MAE: 5.1592
Epoch  50 | Train L1: 8.6742 | Test MAE: 5.0236
Epoch  60 | Train L1: 8.2317 | Test MAE: 4.9965

LSTM — MAE: 4.988 | RMSE: 8.634


In [25]:
# Step 4.5-4.6 — Training Curves & Sequence Visualizations

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Sequence Model Training Curves — Train 18238", fontsize=13, fontweight="bold")

axes[0].plot(tcn_train_losses,  label="TCN",  color="#4C72B0")
axes[0].plot(lstm_train_losses, label="LSTM", color="#DD8452")
axes[0].set_title("Training Loss (L1) per Epoch")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("L1 Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)

sample_n = min(100, len(tcn_actuals))
axes[1].plot(range(sample_n), tcn_actuals[:sample_n],  label="Actual",  color="black",    linewidth=1.3)
axes[1].plot(range(sample_n), tcn_preds[:sample_n],   label="TCN Pred", color="#4C72B0",  linewidth=1.2, linestyle="--")
axes[1].plot(range(sample_n), lstm_preds[:sample_n],  label="LSTM Pred",color="#DD8452",  linewidth=1.2, linestyle=":")
axes[1].set_title("Actual vs Sequence Predictions (first 100 samples)")
axes[1].set_xlabel("Sample"); axes[1].set_ylabel("Delay (min)")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../outputs/plots/sequence_models_eval.png", dpi=120, bbox_inches="tight")
plt.show()


C:\Users\arjun\AppData\Local\Temp\ipykernel_11080\1407575556.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [26]:
# Step 4.7 — Final Comparative Analysis

xgb_mae_seq  = float(np.mean(np.abs(y_seq_te - xgb_best.predict(
    xgb.DMatrix(X_seq_te[:, -1, :], feature_names=usable_feature_cols)))))
xgb_rmse_seq = float(np.sqrt(np.mean((y_seq_te - xgb_best.predict(
    xgb.DMatrix(X_seq_te[:, -1, :], feature_names=usable_feature_cols))) ** 2)))

final_comparison = pd.DataFrame([
    {"model": "Persistence Baseline",  "mae": float(results_df.loc[results_df.model=="persistence", "mae"].values[0]),
                                          "rmse": float(results_df.loc[results_df.model=="persistence", "rmse"].values[0])},
    {"model": "Linear Regression",     "mae": float(results_df.loc[results_df.model=="linear_lstsq", "mae"].values[0]),
                                          "rmse": float(results_df.loc[results_df.model=="linear_lstsq", "rmse"].values[0])},
    {"model": "XGBoost (tuned)",        "mae": xgb_mae_seq,  "rmse": xgb_rmse_seq},
    {"model": "TCN",                    "mae": tcn_mae,       "rmse": tcn_rmse},
    {"model": "LSTM",                   "mae": lstm_mae,      "rmse": lstm_rmse},
]).sort_values("mae").reset_index(drop=True)

print("=== FINAL MODEL COMPARISON ===")
display(final_comparison.style.format({"mae": "{:.3f}", "rmse": "{:.3f}"}))

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 5))
colors_bar = ["#95A5A6", "#3498DB", "#E67E22", "#1ABC9C", "#9B59B6"]
bars = ax.barh(final_comparison["model"][::-1], final_comparison["mae"][::-1],
               color=colors_bar[::-1], edgecolor="white", height=0.55)
for bar, val in zip(bars, final_comparison["mae"][::-1]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f"{val:.2f}", va="center", fontsize=10)
ax.set_title("Model Comparison — MAE (lower is better)", fontsize=13, fontweight="bold")
ax.set_xlabel("MAE (minutes)"); ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("../outputs/plots/final_model_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

print()
print("Key Findings:")
best_model = final_comparison.iloc[0]
print(f"  Best model by MAE: {best_model['model']}  (MAE={best_model['mae']:.3f})")

xgb_row_f = final_comparison[final_comparison["model"] == "XGBoost (tuned)"].iloc[0]
tcn_row_f = final_comparison[final_comparison["model"] == "TCN"].iloc[0]
lstm_row_f = final_comparison[final_comparison["model"] == "LSTM"].iloc[0]
print(f"  Does deep learning outperform XGBoost? {'Yes' if min(tcn_row_f.mae, lstm_row_f.mae) < xgb_row_f.mae else 'No (XGBoost wins)'}")
print(f"  TCN vs LSTM: {'TCN better' if tcn_row_f.mae <= lstm_row_f.mae else 'LSTM better'}")


=== FINAL MODEL COMPARISON ===


,model,mae,rmse
0,TCN,4.778,8.105
1,LSTM,4.988,8.634
2,Linear Regression,7.160,10.277
3,Persistence Baseline,8.375,14.190
4,XGBoost (tuned),14.405,22.619



Key Findings:
  Best model by MAE: TCN  (MAE=4.778)
  Does deep learning outperform XGBoost? Yes
  TCN vs LSTM: TCN better


C:\Users\arjun\AppData\Local\Temp\ipykernel_11080\2266120638.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [27]:
# Save Sequence Models
import os

torch.save(tcn_model.state_dict(),  "../outputs/models/tcn_train18238.pt")
torch.save(lstm_model.state_dict(), "../outputs/models/lstm_train18238.pt")

with open("../outputs/models/scaler_seq.pkl", "wb") as f:
    pickle.dump(scaler_seq, f)

final_comparison.to_csv("../outputs/final_model_comparison.csv", index=False)

print("All artifacts saved to ../outputs/models/")


All artifacts saved to ../outputs/models/


# Summary

All phases complete:
- **Phase 1**: Data cleaning, station mapping, feature engineering (23 features, no leakage).
- **Phase 2**: Persistence, Rolling Mean, Linear Regression, Ridge baselines with time-based split.
- **Phase 3**: XGBoost with Optuna tuning, SHAP importance, recursive multi-station forecasting, confidence bands, error analysis, Flask API.
- **Phase 4**: TCN and LSTM sequence models trained on sliding windows, training curves, and final cross-model comparison.

See `../outputs/plots/` for all visualisations and `../outputs/models/` for serialised artifacts.